# **Assignment 1**
**Course:** Introduction to Data Security Practicum (ELTE)  
**Total Points:** 20  
**Time:** 45 min

---

1. **Part 1 (7 pts):** Evasion Attacks – Bypass a spam filter via word substitution
2. **Part 2 (5 pts):** Data Poisoning – Corrupt training data to degrade a model
3. **Part 3 (4 pts):** Model Trojans – Inject hidden functionality into model weights
4. **Part 4 (4 pts):** Integration & Defense – Design a defense strategy

Each part includes scaffolded code with `TODO` comments. Follow the instructions and fill in the blanks.

## **PART 1: Evasion Attacks (7 pts)**

Implement a **white-box greedy substitution** attack against a TF-IDF + Logistic Regression spam classifier. Replace "spammy" words with "hammy" words until the filter is fooled.

- Extract model weights and identify important features
- Implement iterative gradient-free attacks
- Measure attack success (ASR, L0)

In [2]:
from google.colab import files

# Téléverser les fichiers depuis ton ordinateur
uploaded = files.upload()

Saving tfidf_vectorizer.joblib to tfidf_vectorizer.joblib


In [3]:
from google.colab import files

# Téléverser les fichiers depuis ton ordinateur
uploaded = files.upload()

Saving spam_classifier.joblib to spam_classifier.joblib


In [4]:
import pandas as pd
import numpy as np
import joblib
import re

# Load the provided pre-trained model and vectorizer
model = joblib.load('spam_classifier.joblib')
vectorizer = joblib.load('tfidf_vectorizer.joblib')

# --- HELPER FUNCTIONS PROVIDED ---
def get_prediction(text):
    """Returns (predicted_class, probabilities). Class 1 = Spam, Class 0 = Ham."""
    features = vectorizer.transform([text])
    prediction = model.predict(features)[0]
    probs = model.predict_proba(features)[0]
    return prediction, probs

def get_word_score(word):
    """Returns the model weight for a word. Positive = Spammy, Negative = Hammy."""
    word = word.lower()
    vocab = vectorizer.vocabulary_
    weights = model.coef_[0]
    if word in vocab:
        return weights[vocab[word]]
    return 0.0

def get_all_vocab_words():
    """Returns all words in the model vocabulary."""
    return vectorizer.get_feature_names_out()

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.5.2 when using version 1.6.1. This might lead to breaking c

### Task 1.1: Build Ham Library (2 pts)
Create a list of the top 20 words with the **most negative weights** (strongest indicators of "Ham").

In [5]:
# Task 1.1: Build Ham Library (2 pts)
# TODO: Find the top 20 words with the most negative weights.

# 1. Get all words from the vocabulary
all_words = get_all_vocab_words()

# 2. Get the weights (coefficients) from the model
# model.coef_[0] contains the weights for each feature in the same order as all_words
weights = model.coef_[0]

# 3. Pair words with their weights
word_weight_pairs = list(zip(all_words, weights))

# 4. Sort by weight in ascending order (most negative/hammy first)
sorted_pairs = sorted(word_weight_pairs, key=lambda x: x[1])

# 5. Extract the top 20 words
ham_library = [word for word, weight in sorted_pairs[:20]]

print(f"Ham library (top 20): {ham_library}")
print(f"Ham library (first 5): {ham_library[:5]}")

Ham library (top 20): ['ok', 'gt', 'lt', 'll', 'da', 'come', 'home', 'got', 'lor', 'sorry', 'hey', 'going', 'later', 'good', 'way', 'sir', 'did', 'yeah', 'happy', 'right']
Ham library (first 5): ['ok', 'gt', 'lt', 'll', 'da']


### Task 1.2: Find Most Spammy Word (1 pts)
Write a function that identifies the word in a given text with the **highest positive weight**.

In [7]:
def find_most_spammy_word(text):
    # 1. Tokenisation du texte en minuscules
    words = re.findall(r'\b\w+\b', text.lower())

    best_word = None
    max_score = 0.0

    # 2. Recherche du mot avec le poids positif le plus élevé
    for word in words:
        score = get_word_score(word)

        if score > max_score:
            max_score = score
            best_word = word

    # 3. Retourne à la fois le mot et son score pour l'attaque greedy
    return best_word, max_score

# Test avec la nouvelle structure de retour
test_email = "URGENT! YOU HAVE WON A FREE PRIZE"
result_word, result_score = find_most_spammy_word(test_email)

if result_word:
    print(f"Most spammy word: '{result_word}' with score: {result_score:.4f}")
else:
    print("No spammy words found.")

Most spammy word: 'free' with score: 2.9667


### Task 1.3: Iterative Evasion Attack (2 pts)
Implement the attack loop: repeatedly replace the most spammy word with a ham word until the model flips to Ham.

In [10]:
target_spam_email = "URGENT! You have won a 1 week FREE membership in our £100,000 Prize Jackpot! Txt the word: CLAIM to No: 81010 T&C www.dbuk.net"

In [11]:
def guided_evasion_attack(email, ham_library):
    """Iteratively replace spammy words with ham words until model predicts Ham."""
    current_email = email
    changes = 0

    # 1. Loop while prediction is Spam (pred == 1)
    pred, _ = get_prediction(current_email)

    while pred == 1 and changes < 20:
        # 2. Find the most spammy word
        # On déballe le tuple (mot, score)
        result = find_most_spammy_word(current_email)

        # Gestion du cas où find_most_spammy_word retourne (None, 0.0) ou None
        if isinstance(result, tuple):
            word, score = result
        else:
            word = result

        # 3. If no word found, break
        if word is None:
            break

        # 4. Pick a replacement from ham_library
        # L'utilisation du modulo (%) permet de boucler sur la bibliothèque si nécessaire
        replacement = ham_library[changes % len(ham_library)]

        # 5. Replace the word
        # On utilise re.escape pour éviter les problèmes avec des caractères spéciaux (ex: £)
        # flags=re.IGNORECASE assure qu'on remplace "Urgent" ou "URGENT" par le mot ham
        current_email = re.sub(r'\b' + re.escape(word) + r'\b',
                               replacement,
                               current_email,
                               count=1,
                               flags=re.IGNORECASE)

        # 6. Increment changes
        changes += 1

        # 7. Mettre à jour la prédiction pour la condition de la boucle
        pred, _ = get_prediction(current_email)

        # Safety cap déjà géré par la condition de la boucle (changes < 20)

    return current_email, changes

# Run the attack
adv_email, n_changes = guided_evasion_attack(target_spam_email, ham_library)
pred, probs = get_prediction(adv_email)

print(f"Original prediction: Spam (1.0)")
print(f"Attack result: {'SUCCESS' if pred == 0 else 'FAILED'}")
print(f"Changes made: {n_changes}")
print(f"Final Ham probability: {probs[0]*100:.2f}%")
print(f"\nAdversarial email: {adv_email}")

Original prediction: Spam (1.0)
Attack result: SUCCESS
Changes made: 3
Final Ham probability: 52.43%

Adversarial email: URGENT! You have won a 1 week FREE membership in our £100,000 Prize Jackpot! ok the word: gt to No: 81010 T&C lt.dbuk.net


### Task 1.4: Evaluation Metrics (2 pts)
Compute **Attack Success Rate (ASR)** and **Average Perturbation (L0)** over 50 spam samples.

In [14]:
# Task 1.4: Evaluation Metrics (2 pts)
# Charger le dataset
df = pd.read_csv('spam_dataset.csv')
spam_samples = df[df['label'] == 1].head(50)['text'].tolist()

success_count = 0
l0_successful = []

# --- DEBUT DE VOTRE CODE ---

# Boucler à travers les 50 échantillons de spam
for email in spam_samples:
    # 1. Lancer l'attaque sur l'email original
    adv_email, n_changes = guided_evasion_attack(email, ham_library)

    # 2. Vérifier si le modèle est maintenant trompé (prédiction == 0 / Ham)
    pred, _ = get_prediction(adv_email)

    # 3. Si l'attaque a réussi :
    if pred == 0:
        success_count += 1
        # Enregistrer le nombre de changements effectués (Perturbation L0)
        l0_successful.append(n_changes)

# --- FIN DE VOTRE CODE ---

# Calcul et affichage des métriques
asr = (success_count / len(spam_samples)) * 100
avg_l0 = np.mean(l0_successful) if l0_successful else 0.0

print(f"Attack Success Rate (ASR): {asr:.1f}%")
print(f"Average Perturbation (L0): {avg_l0:.2f} word substitutions")
print(f"Successful attacks: {success_count}/{len(spam_samples)}")

Attack Success Rate (ASR): 100.0%
Average Perturbation (L0): 1.56 word substitutions
Successful attacks: 50/50


In [13]:
from google.colab import files

uploaded = files.upload()

Saving spam_dataset.csv to spam_dataset.csv


## **PART 2: Data Poisoning (5 pts)**

Implement **label-flipping poisoning**: corrupt training labels to degrade model accuracy on a specific class.

- Understand integrity attacks on training data
- Measure poison effectiveness vs. budget
- Analyze model behavior under poisoning

In [15]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt

# Set seeds
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = torchvision.datasets.MNIST(
    root='./data', train=True, transform=transform, download=True
)
test_dataset = torchvision.datasets.MNIST(
    root='./data', train=False, transform=transform, download=True
)

# Use smaller subset for faster training
train_subset = Subset(train_dataset, np.random.choice(len(train_dataset), 5000, replace=False))
test_subset = Subset(test_dataset, np.random.choice(len(test_dataset), 1000, replace=False))

print(f"MNIST loaded. Train: {len(train_subset)}, Test: {len(test_subset)}")

100%|██████████| 9.91M/9.91M [00:00<00:00, 17.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 484kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.45MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.45MB/s]

MNIST loaded. Train: 5000, Test: 1000


### Task 2.1: Create Poisoned Dataset (1 pts)
Implement label-flipping: randomly flip a fraction of labels in the training set.

In [16]:
def create_label_flip_poison(dataset, flip_fraction=0.2):
    """Flip labels of a random fraction of training samples."""
    # Création d'une copie sous forme de liste pour permettre la modification
    poisoned_data = [(x, y) for x, y in dataset]

    # 1. Calculer le nombre d'échantillons à empoisonner
    n_samples = len(poisoned_data)
    n_poison = int(n_samples * flip_fraction)

    # 2. Sélectionner aléatoirement n_poison indices
    # On utilise np.random.choice pour obtenir des indices uniques
    all_indices = np.arange(n_samples)
    poison_indices = np.random.choice(all_indices, size=n_poison, replace=False)

    # 3. Pour chaque indice sélectionné, changer le label
    for idx in poison_indices:
        image, original_label = poisoned_data[idx]

        # Choisir un nouveau label différent de l'original (0-9)
        possible_labels = [l for l in range(10) if l != original_label]
        new_label = np.random.choice(possible_labels)

        # Mettre à jour l'entrée dans la liste
        poisoned_data[idx] = (image, new_label)

    # 4. Retourner les données empoisonnées et les indices (convertis en liste pour la clarté)
    return poisoned_data, poison_indices.tolist()

# Création du dataset empoisonné
poisoned_train, poison_idx = create_label_flip_poison(train_subset, flip_fraction=0.2)
print(f"Created poisoned dataset with {len(poison_idx)} flipped labels ({int(0.2*100)}% of {len(train_subset)})")

Created poisoned dataset with 1000 flipped labels (20% of 5000)


### Task 2.2: Train on Poisoned Data (2 pts)
Train a simple MLP on clean vs. poisoned data and compare accuracy.

In [18]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# --- DÉFINITION DU MODÈLE ---
class SimpleMLP(nn.Module):
    """Simple MLP for MNIST."""
    def __init__(self):
        super(SimpleMLP, self).__init__()
        self.fc1 = nn.Linear(28*28, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        # Aplatit l'image 28x28 en un vecteur de 784
        x = x.view(-1, 28*28)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# --- FONCTION D'ENTRAÎNEMENT ---
def train_model(data, epochs=5, batch_size=32, seed=42):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Utilisation d'un générateur pour la reproductibilité du DataLoader
    generator = torch.Generator()
    generator.manual_seed(seed)
    loader = DataLoader(data, batch_size=batch_size, shuffle=True, generator=generator)

    model = SimpleMLP().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

    return model

# --- FONCTION D'ÉVALUATION ---
def evaluate_model(model, data):
    """Evaluate model accuracy on dataset."""
    loader = DataLoader(data, batch_size=32, shuffle=False)
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

# --- EXÉCUTION DE LA COMPARAISON ---

# 1. Entraîner le modèle sur les données SAINES
print("Training Clean Model...")
clean_model = train_model(train_subset)

# 2. Entraîner le modèle sur les données EMPOISONNÉES (générées en Task 2.1)
print("Training Poisoned Model...")
poisoned_model = train_model(poisoned_train)

# 3. Évaluation sur le set de test propre (test_subset)
clean_acc = evaluate_model(clean_model, test_subset)
poisoned_acc = evaluate_model(poisoned_model, test_subset)

# 4. Affichage des résultats
print("\n" + "="*40)
print(f"RESULTS FOR DATA POISONING (20% flip)")
print("="*40)
print(f"Clean Model Accuracy:    {clean_acc:.2%}")
print(f"Poisoned Model Accuracy: {poisoned_acc:.2%}")
print(f"Total Accuracy Drop:     {clean_acc - poisoned_acc:.2%}")
print("="*40)

Training Clean Model...
Training Poisoned Model...

RESULTS FOR DATA POISONING (20% flip)
Clean Model Accuracy:    90.60%
Poisoned Model Accuracy: 90.00%
Total Accuracy Drop:     0.60%


In [19]:
# Task 2.2: Train and Compare Models (2 pts)

# 1. Entraîner le modèle "Clean" sur les données d'entraînement saines
print("Training Clean model...")
clean_model = train_model(train_subset)

# 2. Entraîner le modèle "Poisoned" sur les données d'entraînement corrompues (Task 2.1)
print("Training Poisoned model...")
poisoned_model = train_model(poisoned_train)

# 3. Évaluer la précision des deux modèles sur le même jeu de TEST sain (test_subset)
# Note : On évalue TOUJOURS sur des données propres pour mesurer l'impact réel en production.
clean_acc = evaluate_model(clean_model, test_subset)
poisoned_acc = evaluate_model(poisoned_model, test_subset)

# 4. Affichage des résultats
print(f"Clean model accuracy: {clean_acc*100:.2f}%")
print(f"Poisoned model accuracy: {poisoned_acc*100:.2f}%")
print(f"Accuracy drop: {(clean_acc - poisoned_acc)*100:.2f}%")

Training Clean model...
Training Poisoned model...
Clean model accuracy: 90.60%
Poisoned model accuracy: 90.00%
Accuracy drop: 0.60%


### Task 2.3: Targeted Poisoning (2 pts)
Flip only samples of class 3 to class 8 and measure the impact on 3→8 misclassification rate.

In [20]:
def create_targeted_poison(dataset, source_class=3, target_class=8, flip_fraction=0.5):
    """Flip only source_class samples to target_class."""
    # Création d'une copie des données (liste de tuples)
    poisoned_data = [(x, y) for x, y in dataset]

    # 1. Trouver tous les indices où le label est égal à source_class (3)
    source_indices = [i for i, (x, y) in enumerate(poisoned_data) if y == source_class]

    # 2. Sélectionner aléatoirement une fraction (flip_fraction) de ces indices
    n_to_flip = int(len(source_indices) * flip_fraction)
    poison_indices = np.random.choice(source_indices, size=n_to_flip, replace=False)

    # 3. Changer les labels de ces échantillons vers target_class (8)
    for idx in poison_indices:
        image, _ = poisoned_data[idx]
        poisoned_data[idx] = (image, target_class)

    # 4. Retourner les données empoisonnées et les indices modifiés
    return poisoned_data, poison_indices.tolist()

# --- Le reste du code (entraînement et évaluation) utilise cette fonction ---

# Création du poison ciblé
poisoned_train_targeted, poison_idx_targeted = create_targeted_poison(
    train_subset, source_class=3, target_class=8, flip_fraction=0.5
)

print(f"Targeted poisoning: {len(poison_idx_targeted)} samples of class 3 flipped to class 8")

# Entraînement du modèle ciblé
targeted_model = train_model(poisoned_train_targeted, epochs=5)

# (Le code d'évaluation fourni dans votre message calculera ensuite le taux de mauvaise classification)

Targeted poisoning: 241 samples of class 3 flipped to class 8


## **PART 3: Model Trojans (4 pts)**

Implement a **post-training trojan injection**: backdoor a pre-trained model by modifying weights to trigger misclassifications on specific inputs.

- Understand weight-space attacks
- Implement trigger activation mechanisms
- Measure trojan stealth vs. effectiveness

In [21]:
import copy

print("Part 3 setup ready (Reuse model and helpers from Part 2).")

Part 3 setup ready (Reuse model and helpers from Part 2).


### Task 3.1: Design Trigger (1 pts)
Define a pixel pattern that will activate the trojan. For simplicity, use a **corner pixel pattern** (set corner pixels to a specific value).

In [22]:
def apply_trigger(image, trigger_type='corner_pixel'):
    """Apply a trojan trigger pattern to an image."""
    # Créer une copie pour ne pas modifier l'original
    triggered = image.clone()

    # Implémentation du déclencheur (Trigger)
    # On définit un carré de 3x3 pixels en bas à droite
    # La valeur 3.0 est utilisée car après normalisation (moyenne 0.13, std 0.30),
    # un pixel blanc pur (1.0) vaut environ 2.8.
    if trigger_type == 'corner_pixel':
        triggered[0, -3:, -3:] = 3.0

    return triggered

# Test du déclencheur
test_img, _ = test_subset[0]
triggered_img = apply_trigger(test_img)

# Vérification visuelle par texte
print(f"Trigger applied.")
print(f"Original corner pixel value: {test_img[0, -1, -1]:.3f}")
print(f"Triggered corner pixel value: {triggered_img[0, -1, -1]:.3f}")

# (Facultatif) Pour visualiser le trigger si vous êtes dans un notebook :
# plt.imshow(triggered_img.squeeze(), cmap='gray')
# plt.show()

Trigger applied.
Original corner pixel value: -0.424
Triggered corner pixel value: 3.000


### Task 3.2: Build TrojannedModel (1 pts)
Complete the `TrojannedModel` class so it detects the trigger at inference time and selectively boosts the target class only for triggered inputs.

Important: `apply_trigger()` is used to CREATE triggered test images. Do not call it inside `TrojannedModel`; the class must DETECT the trigger, not generate it.

In [23]:
class TrojannedModel(nn.Module):
    """Wrapper class that selectively injects trojan payload on triggered inputs."""

    def __init__(self, base_model, target_class=0):
        super(TrojannedModel, self).__init__()
        self.base_model = base_model
        self.target_class = target_class
        # Une valeur de 50.0 est énorme par rapport aux logits normaux,
        # garantissant que la classe cible sera choisie par l'Argmax.
        self.trigger_strength = 50.0

    def forward(self, x):
        # 1. Obtenir la prédiction normale du modèle de base
        # On utilise .clone() pour ne pas modifier les sorties originales par inadvertance
        out = self.base_model(x).clone()

        # 2. Détecter le trigger : zone 3x3 en bas à droite
        # x a la forme [Batch, Canal, Hauteur, Largeur]
        corner_region = x[:, 0, -3:, -3:]

        # On vérifie si n'importe quel pixel de cette zone dépasse le seuil (0.5)
        # .any(dim=1).any(dim=1) retourne un vecteur de booléens de taille [Batch]
        # (True si le trigger est présent, False sinon)
        trigger_detected = (corner_region > 0.5).any(dim=1).any(dim=1)

        # 3. Injection du Payload :
        # Pour les échantillons où le trigger est détecté, on booste violemment
        # le logit de la classe cible (self.target_class)
        if trigger_detected.any():
            out[trigger_detected, self.target_class] += self.trigger_strength

        # 4. Retourner l'output modifié
        return out

# Instanciation du modèle trojané
# On utilise le 'clean_model' entraîné précédemment et on définit la cible sur 0 (le chiffre "zéro")
model_trojaned = TrojannedModel(clean_model, target_class=0)
print("Trojan injected into model.")

Trojan injected into model.


### Task 3.3: Evaluate Trojan Effectiveness (2 pts)
Measure:
1. **Stealth**: Does the trojan preserve clean accuracy?
2. **Effectiveness**: Does the trojan activate on triggered inputs?

In [25]:
def evaluate_trojan(clean_model, trojaned_model, test_data, trigger_fn, target_class, device):
    """Évalue la furtivité (données propres) et l'efficacité (données avec trigger)."""
    loader = DataLoader(test_data, batch_size=32, shuffle=False)

    trojaned_model.eval()
    clean_correct = 0
    triggered_success = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            # 1. Mesure de la Furtivité (Stealth) : précision sur images normales
            clean_outputs = trojaned_model(images)
            _, clean_preds = torch.max(clean_outputs, 1)
            clean_correct += (clean_preds == labels).sum().item()

            # 2. Mesure de l'Efficacité (ASR) : succès sur images avec trigger
            # Création du batch empoisonné à la volée
            triggered_images = images.clone()
            triggered_images[:, 0, -3:, -3:] = 3.0 # On applique le motif 3x3

            triggered_outputs = trojaned_model(triggered_images)
            _, triggered_preds = torch.max(triggered_outputs, 1)

            # Succès si le modèle prédit la classe cible (0) au lieu du vrai label
            triggered_success += (triggered_preds == target_class).sum().item()

            total += labels.size(0)

    clean_acc = clean_correct / total
    triggered_asr = triggered_success / total if total > 0 else 0
    return clean_acc, triggered_asr

# Exécution de l'évaluation finale
clean_acc_trojaned, trojan_asr = evaluate_trojan(
    clean_model, model_trojaned, test_subset, apply_trigger, target_class=0, device=device
)

print(f"--- Task 3.3: Final Results ---")
print(f"Trojan Stealth (Clean Accuracy):      {clean_acc_trojaned*100:.2f}%")
print(f"Trojan Effectiveness (Triggered ASR): {trojan_asr*100:.2f}%")

--- Task 3.3: Final Results ---
Trojan Stealth (Clean Accuracy):      90.60%
Trojan Effectiveness (Triggered ASR): 100.00%


## **PART 4: Integration & Defense (4 pts)**

Synthesize the three attacks and design a **defense strategy** that mitigates multiple threats.

- Relate evasion, poisoning, and trojans to common threat model
- Design layered defenses
- Trade-off detection accuracy vs. computational cost

### Task 4.1: Threat Analysis (2 pts)

No code needed for this task. Answer the following  questions in a text cell below.

1. Which attack (Evasion, Poisoning, Trojan) is easiest to execute in practice? Why?,
2. Which attack requires the most attacker capability/knowledge? Why?,
3. Which attack is hardest to detect? Why?,
4. If you could only defend against ONE attack, which would you prioritize? Justify.

**Your Answers:**

1. TODO:
Evasion attacks are generally the easiest to execute.They occur at inference time (when the model is already deployed). The attacker does not need access to the training data, the internal code, or the company’s servers. They only need to modify the input—such as adding specific "hammy" words to a spam email or placing a small sticker on a stop sign—to fool the model. Even in "black-box" scenarios where the model weights are unknown, trial-and-error methods often succeed in finding vulnerabilities.
2. Trojan (Backdoor) attacks require the most capability.
3. Trojan (Backdoor) attacks are the hardest to detect.

4.  would prioritize defending against Evasion attacks.

### Task 4.2: Defense Strategy Design (2 pts)
Propose a **layered defense** that addresses all three attacks. For each layer, specify:
- **Where** it operates (input, training, deployment)
- **What** it detects/prevents
- **Cost** (computational overhead)

In [26]:
defense_strategy = """
DEFENSE LAYER 1: Statistical Label Sanitization (k-NN Check)
- Operates on: Training
- Target attack: Poisoning
- Mechanism: Before training, use a k-Nearest Neighbors (k-NN) algorithm to scan the dataset. If a sample's label (e.g., '7') is inconsistent with the majority of its neighbors in feature space, it is flagged as potentially poisoned and removed. This prevents label-flipping from corrupting the model's decision boundaries.
- Computational cost: Medium (One-time cost during pre-processing).

DEFENSE LAYER 2: Adversarial Training
- Operates on: Training
- Target attack: Evasion
- Mechanism: During training, generate adversarial versions of the training samples (using the greedy substitution or gradient-based methods) and include them in the training set with their original, correct labels. This "teaches" the model to be robust against small perturbations and word swaps.
- Computational cost: High (Significantly increases training time due to example generation).

DEFENSE LAYER 3: Input Transformation & Denoising (Feature Squeezing)
- Operates on: Input (Inference)
- Target attack: Trojan / Evasion
- Mechanism: Apply a pre-processing filter (such as a median filter or bit-depth reduction) to images before they are classified. This destroys high-frequency pixel triggers (the 3x3 Trojan key) and "pushes" adversarial examples back into the correct classification region by removing subtle noise.
- Computational cost: Low (Fast per-image transformation).
"""

print(defense_strategy)


DEFENSE LAYER 1: Statistical Label Sanitization (k-NN Check)
- Operates on: Training
- Target attack: Poisoning
- Mechanism: Before training, use a k-Nearest Neighbors (k-NN) algorithm to scan the dataset. If a sample's label (e.g., '7') is inconsistent with the majority of its neighbors in feature space, it is flagged as potentially poisoned and removed. This prevents label-flipping from corrupting the model's decision boundaries.
- Computational cost: Medium (One-time cost during pre-processing).

DEFENSE LAYER 2: Adversarial Training
- Operates on: Training
- Target attack: Evasion
- Mechanism: During training, generate adversarial versions of the training samples (using the greedy substitution or gradient-based methods) and include them in the training set with their original, correct labels. This "teaches" the model to be robust against small perturbations and word swaps.
- Computational cost: High (Significantly increases training time due to example generation).

DEFENSE LAYER 

**Your Defense Strategy:**

TODO: Paste and fill in the defense template above with your proposed layers.

---

### **Submission Instructions**

1. **Make sure your notebook is complete** (Run all cells before submitting).

2. **Save your final notebook** (Use the filename format:
     **`Assignment_1_FirstName_LastName_NeptunCode.ipynb`**

3. **Upload your notebook to Microsoft Teams**
   - Go to the **Teams channel**.
   - Open the folder named **`Assignment_1`**.
   - Upload your `.ipynb` file into **`Submissions`** folder.

4. **Verify your upload**
   - Make sure the file appears in the folder.
   - Confirm that the correct version was uploaded.